# Create Finnish Foundation for Cardiovascular Research Awards (Sydäntutkimussäätiö)

Creates the research grants of the Finnish Foundation for Cardiovascular
Research (Sydäntutkimussäätiö) from the foundation's own annual grant lists.

**Prerequisite:** `scripts/local/sydantutkimussaatio_to_s3.py` (uploads the parquet).

**Source:** https://www.sydantutkimussaatio.fi/apurahat/myonnetyt-apurahat links one
PDF per grant year, 2010-2020 and 2022-2026 (no 2021 list is published; earlier years
are only "available from the office"). Every PDF lists grants under fund headers
('<fund> rahasto N kpl, TOTAL €'); each grant gives the grantee's academic title and
name (name set in bold), amount, institution and city, and project title. The script
reconciles every fund against its printed grant count and euro total; all 16 PDFs
reconcile (one 2011 header prints 34 for 35 listed grants, euro total exact).
research.fi (ladder item 0) does not carry this foundation.

**1,423 grants, 2010-2026, EUR 35.5M; 100% title, amount, named grantee, institution**
(local run 2026-10-01).

**Scope:** every grant in the lists is kept: personal research grants from the
foundation's general fund and its ~35 named funds (incl. the 2010/2011/2020 jubilee
funds and Finnish Heart Association funds the foundation distributes). The
foundation's recognition prizes (Tunnustuspalkinto, Soisalo prize) are separate
pages and not included.

**Field mapping:** `display_name` = project title; `funder_scheme` = fund name;
`start_year` = grant year (decisions each April/May); `start_date`/`end_date` NULL (not
published); `lead_investigator` = grantee (given/family split by the canonical
`split_name`), affiliation = institution line as printed; amounts EUR.

**`funder_award_id` (2.1.1):** the lists print no grant number. Citing works write
6-digit application numbers (`200061`, `230012` style) in the ~217 acknowledgement
shell rows, which the foundation never publishes, so a synthetic
`SYD-<year>-<family>-<given>` key is used (`-b`, `-c` on a same-year repeat, e.g. one
person funded from two funds); 0 shells collapse.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320323333`
- display_name / ror_id / doi: from `openalex.funders.funders` (ror 020apt798, doi 10.13039/501100005633)

**Priority:** `568` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.sydantutkimussaatio_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/sydantutkimussaatio/sydantutkimussaatio_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_grants, COUNT(DISTINCT funder_award_id) as distinct_ids,
       MIN(grant_year) as min_year, MAX(grant_year) as max_year, ROUND(SUM(amount), 0) as total_eur
FROM openalex.awards.sydantutkimussaatio_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.sydantutkimussaatio_raw;

In [ ]:
%sql
SELECT grant_year, COUNT(*) AS n, ROUND(SUM(amount), 0) AS eur
FROM openalex.awards.sydantutkimussaatio_raw GROUP BY 1 ORDER BY 1;

## Step 1.6: Funder existence check (Path A)

In [ ]:
%sql
SELECT assert_true(COUNT(*) = 1, 'F4320323333 must have exactly one openalex.funders.funders row') AS funder_ok,
       MAX(display_name) AS display_name, MAX(ror_id) AS ror_id, MAX(doi) AS doi
FROM openalex.funders.funders
WHERE funder_id = 4320323333;

## Step 2: Create Sydäntutkimussäätiö Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.sydantutkimussaatio_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320323333  -- Sydäntutkimussäätiö
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    COALESCE(NULLIF(TRIM(g.title), ''),
             CONCAT('Finnish Foundation for Cardiovascular Research grant: ', g.grantee, ' (', g.grant_year, ')')) as display_name,
    CAST(NULL AS STRING) as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    TRY_CAST(g.amount AS DOUBLE) as amount,
    'EUR' as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'research' as funding_type,
    NULLIF(TRIM(g.fund), '') as funder_scheme,
    'sydantutkimussaatio_grant_lists' as provenance,
    CAST(NULL AS DATE) as start_date,
    CAST(NULL AS DATE) as end_date,
    TRY_CAST(g.grant_year AS INT) as start_year,
    CAST(NULL AS INT) as end_year,
    -- struct field order MUST match openalex_awards_raw: given, family, orcid, role_start, affiliation.
    CASE WHEN NULLIF(TRIM(g.lead_family_name), '') IS NOT NULL THEN named_struct(
        'given_name', NULLIF(TRIM(g.lead_given_name), ''),
        'family_name', NULLIF(TRIM(g.lead_family_name), ''),
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', named_struct(
            'name', NULLIF(TRIM(g.institution), ''),
            'country', CAST(NULL AS STRING),
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        )
    ) END as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.sydantutkimussaatio_raw g
CROSS JOIN src_funder f
WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != '';

## Shape check (2.1.1)
Synthetic `SYD-<year>-<family slug>-<given slug>[-b..]` keys only.

In [ ]:
%sql
SELECT assert_true(COUNT(*) = 0, 'unexpected funder_award_id shape') AS shape_ok
FROM openalex.awards.sydantutkimussaatio_awards
WHERE NOT funder_award_id RLIKE '^SYD-(19|20)[0-9]{2}-[a-z0-9-]+$';

In [ ]:
%sql
SELECT assert_true(COUNT(*) = COUNT(DISTINCT id), 'duplicate award ids') AS unique_ok, COUNT(*) AS n
FROM openalex.awards.sydantutkimussaatio_awards;

## Step 3: Insert into openalex_awards_raw

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'sydantutkimussaatio_grant_lists' AND priority = 568;

-- Priority 568: direct-from-funder ingest of the foundation's own annual grant lists.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    568 as priority
FROM openalex.awards.sydantutkimussaatio_awards;

## Verification Queries

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, start_year,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name
FROM openalex.awards.sydantutkimussaatio_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_eur
FROM openalex.awards.sydantutkimussaatio_awards GROUP BY 1 ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_eur
FROM openalex.awards.sydantutkimussaatio_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.sydantutkimussaatio_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.sydantutkimussaatio_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- 6.3 / 6.7 coverage.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_year) as has_start_year,
    COUNT(lead_investigator) as has_pi
FROM openalex.awards.sydantutkimussaatio_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'sydantutkimussaatio_grant_lists'
GROUP BY provenance, priority;